### Task 1.

Раньше мы оценивали эксперимент "Refactoring backend", предполагая, что данные времени работы бэкенда независимые. Теперь мы можем корректно оценить этот эксперимент, даже если данные зависели бы от пользователей.

Оцените эксперимент "Refactoring backend" с использованием линеаризации в предположении, что данные пользователей зависимы.

Данные эксперимента "Refactoring backend": 2022-04-13/2022-04-13T12_df_web_logs.csv и 2022-04-13/experiment_users.csv. Эксперимент проводился с 2022-04-05 по 2022-04-12. Метрика — среднее время обработки запроса.

В качестве ответа введите p-value с точность до 4-го знака после точки.

**Ответ**

0.0442

### Task 2.

Реализуйте функцию calculate_linearized_metrics.

In [ ]:
import pandas as pd


def calculate_linearized_metrics(control_metrics, pilot_metrics):
    """Считает значения линеаризованной метрики.

    Нужно вычислить параметр kappa (коэффициент в функции линеаризации) по данным из
    control_metrics и использовать его для вычисления линеаризованной метрики.

    :param control_metrics (pd.DataFrame): датафрейм со значениями метрики контрольной группы.
        Значения в столбце 'user_id' не уникальны.
        Измерения для одного user_id считаем зависимыми, а разных user_id - независимыми.
        columns=['user_id', 'metric']
    :param pilot_metrics (pd.DataFrame): датафрейм со значениями метрики экспериментальной группы.
        Значения в столбце 'user_id' не уникальны.
        Измерения для одного user_id считаем зависимыми, а разных user_id - независимыми.
        columns=['user_id', 'metric']
    :return lin_control_metrics, lin_pilot_metrics: датафреймы контрольной и экспериментальногй
        групп со значениями линеаризованной метрики.
        columns=['user_id', 'metric']
    """
    # YOUR_CODE_HERE

**Пример**

In [ ]:
control_metrics = pd.DataFrame({'user_id': [1, 1, 2], 'metric': [3, 5, 7],})
pilot_metrics = pd.DataFrame({'user_id': [3, 3], 'metric': [3, 6], })
lin_control_metrics, lin_pilot_metrics = calculate_linearized_metrics(
    control_metrics, pilot_metrics
)
# lin_control_metrics = pd.DataFrame({'user_id': [1, 2], 'metric': [-2, 2]})
# lin_pilot_metrics = pd.DataFrame({'user_id': [3,], 'metric': [-1,]})

**Решение**

In [2]:
import pandas as pd
import numpy as np

def calculate_linearized_metrics(control_metrics, pilot_metrics, control_user_ids=None, pilot_user_ids=None
    ):
        """Считает значения метрики отношения.

        Нужно вычислить параметр kappa (коэффициент в функции линеаризации) по данным из
        control_metrics и использовать его для вычисления линеаризованной метрики.

        :param control_metrics (pd.DataFrame): датафрейм со значениями метрики контрольной группы.
            Значения в столбце 'user_id' не уникальны.
            Измерения для одного user_id считаем зависимыми, а разных user_id - независимыми.
            columns=['user_id', 'metric']
        :param pilot_metrics (pd.DataFrame): датафрейм со значениями метрики экспериментальной группы.
            Значения в столбце 'user_id' не уникальны.
            Измерения для одного user_id считаем зависимыми, а разных user_id - независимыми.
            columns=['user_id', 'metric']
        :param control_user_ids (list): список id пользователей контрольной группы, для которых
            нужно рассчитать метрику. Если None, то использовать пользователей из control_metrics.
            Если для какого-то пользователя нет записей в таблице control_metrics, то его
            линеаризованная метрика равна нулю.
        :param pilot_user_ids (list): список id пользователей экспериментальной группы, для которых
            нужно рассчитать метрику. Если None, то использовать пользователей из pilot_metrics.
            Если для какого-то пользователя нет записей в таблице pilot_metrics, то его
            линеаризованная метрика равна нулю.
        :return lin_control_metrics, lin_pilot_metrics: columns=['user_id', 'metric']
        """

        control_data = control_metrics.groupby('user_id').agg(x=('metric', 'sum'), y=('metric', 'count')).reset_index()
        pilot_data = pilot_metrics.groupby('user_id').agg(x=('metric', 'sum'), y=('metric', 'count')).reset_index()

        kappa = np.sum(control_data['x']) / np.sum(control_data['y'])

        control_users = pd.Series(control_user_ids).to_frame(name='user_id')
        pilot_users = pd.Series(pilot_user_ids).to_frame(name='user_id')

        control_data = control_users.merge(control_data, how='left', on='user_id').fillna(0).copy() if control_user_ids is not None else control_data.copy()
        pilot_data = pilot_users.merge(pilot_data, how='left', on='user_id').fillna(0).copy()  if pilot_user_ids is not None else pilot_data.copy()

        control_data['metric'] = control_data['x'] - kappa * control_data['y']
        pilot_data['metric'] = pilot_data['x'] - kappa * pilot_data['y']

        return control_data[['user_id', 'metric']], pilot_data[['user_id', 'metric']]

Проверка

In [3]:
control_metrics = pd.DataFrame({'user_id': [1, 1, 2], 'metric': [3, 5, 7],})
pilot_metrics = pd.DataFrame({'user_id': [3, 3], 'metric': [3, 6], })
lin_control_metrics, lin_pilot_metrics = calculate_linearized_metrics(
    control_metrics, pilot_metrics
)
# lin_control_metrics = pd.DataFrame({'user_id': [1, 2], 'metric': [-2, 2]})
# lin_pilot_metrics = pd.DataFrame({'user_id': [3,], 'metric': [-1,]})

In [4]:
lin_control_metrics

,user_id,metric
0,1,-2.0
1,2,2.0


In [5]:
lin_pilot_metrics

,user_id,metric
0,3,-1.0
